<a href="https://colab.research.google.com/github/abrarhossam51-max/Phantoms-W2/blob/main/W2D5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 db_design.md


# Appointment Booking System — Database Design

## 1. Assumptions

1. The system supports multiple customers and service providers.
2. Each appointment belongs to one customer and one service provider.
3. A customer can book multiple appointments.
4. A provider can have multiple appointments, but two active appointments must not overlap.
5. Cancelled appointments remain in the database for history and reporting.
6. Each appointment has a start time, end time, status, and creation timestamp.
7. The system stores timestamps in a consistent format.
8. Authentication and payment processing are outside the scope of this database design.

## 2. Database Schema

### customers
- customer_id: INTEGER, Primary Key
- full_name: TEXT, NOT NULL
- email: TEXT, UNIQUE, NOT NULL
- created_at: TEXT, NOT NULL

### providers
- provider_id: INTEGER, Primary Key
- full_name: TEXT, NOT NULL
- service_type: TEXT, NOT NULL
- email: TEXT, UNIQUE, NOT NULL

### appointments
- appointment_id: INTEGER, Primary Key
- customer_id: INTEGER, Foreign Key -> customers.customer_id
- provider_id: INTEGER, Foreign Key -> providers.provider_id
- start_time: TEXT, NOT NULL
- end_time: TEXT, NOT NULL
- status: TEXT, NOT NULL
- created_at: TEXT, NOT NULL

## 3. Relationships

- One customer can have many appointments.
- One provider can have many appointments.
- Each appointment belongs to exactly one customer and one provider.

## 4. Design Decisions

- Separate customer and provider tables avoid duplicated information.
- Foreign keys maintain referential integrity.
- Appointment statuses include BOOKED, COMPLETED, and CANCELLED.
- Cancelled appointments are retained for audit history.
- An index on provider_id and start_time helps appointment lookups.
- Time-overlap prevention requires an additional validation
  when a booking is created or updated.
- A database transaction should be used when checking availability
  and creating an appointment to reduce booking conflicts.

## 5. Future Improvements

- Add a services table if providers offer multiple services.
- Add an appointment_status_history table to track every status change.
- Add availability schedules and holidays.
- Add notifications and payment records if required.

 debug_solution.sql

-- Bug explanation:
-- The WHERE condition on the right table removes customers
-- whose orders do not match COMPLETED, including customers
-- with no orders. This defeats the purpose of LEFT JOIN.
--
-- Fix:
-- Move the status condition into the ON clause so every
-- customer remains in the result.
--
-- GROUP BY:
-- Include customer_id and customer_name explicitly for
-- compatibility with strict SQL grouping rules.

SELECT
    c.customer_id,
    c.customer_name,
    COALESCE(SUM(o.total_amount), 0) AS total_spent
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
   AND o.order_status = 'COMPLETED'
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY
    c.customer_id;

Src/automation_pipeline.py


import os
import sqlite3
from pathlib import Path
from datetime import datetime

import requests

BASE_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = BASE_DIR / "data"
DB_PATH = DATA_DIR / "store.db"

API_KEY = os.getenv("OPENWEATHER_API_KEY")
CITY = os.getenv("WEATHER_CITY", "Alexandria")
WEBHOOK_URL = os.getenv("WEBHOOK_URL")

API_URL = "https://api.openweathermap.org/data/2.5/weather"


def setup_database():
    DATA_DIR.mkdir(exist_ok=True)

    with sqlite3.connect(DB_PATH) as connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS weather_data (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                city TEXT NOT NULL,
                temperature REAL NOT NULL,
                description TEXT NOT NULL,
                created_at TEXT NOT NULL
            )
        """)


def run_pipeline():
    if not API_KEY or not WEBHOOK_URL:
        raise RuntimeError(
            "Set OPENWEATHER_API_KEY and WEBHOOK_URL first."
        )

    # Step 1: Fetch API data
    response = requests.get(
        API_URL,
        params={
            "q": CITY,
            "appid": API_KEY,
            "units": "metric"
        },
        timeout=15
    )
    response.raise_for_status()
    raw_data = response.json()

    # Step 2: Prepare one payload for storage and webhook
    payload = {
        "city": raw_data["name"],
        "temperature": raw_data["main"]["temp"],
        "description": raw_data["weather"][0]["description"],
        "created_at": datetime.now().isoformat(timespec="seconds")
    }

    # Step 3: Store the data
    with sqlite3.connect(DB_PATH) as connection:
        connection.execute("""
            INSERT INTO weather_data
            (city, temperature, description, created_at)
            VALUES (?, ?, ?, ?)
        """, (
            payload["city"],
            payload["temperature"],
            payload["description"],
            payload["created_at"]
        ))

    # Step 4: Send the same payload to the webhook
    webhook_response = requests.post(
        WEBHOOK_URL,
        json=payload,
        timeout=15
    )
    webhook_response.raise_for_status()

    return payload


if __name__ == "__main__":
    setup_database()
    result = run_pipeline()
    print("Pipeline completed:", result)

Main.py


import time
from src.automation_pipeline import setup_database, run_pipeline


def main():
    setup_database()
    cycle = 1

    while True:
        print(f"\nStarting cycle {cycle}...")

        try:
            run_pipeline()
            print(f"تم تنفيذ الدورة رقم {cycle} بنجاح")
        except Exception as error:
            print(f"الدورة رقم {cycle} فشلت: {error}")

        cycle += 1
        time.sleep(30)


if __name__ == "__main__":
    main()

Readme.md


# Phantoms AI — Week 2

## Project Overview

This project combines API integration, webhook communication,
SQLite storage, automation, database design, SQL debugging,
and a simple database test.

## Project Structure

- `data/`: SQLite database.
- `src/`: API and pipeline logic.
- `tests/`: Basic verification tests.
- `db_design.md`: Appointment booking database design.
- `debug_solution.sql`: Corrected sales analysis query.
- `main.py`: Unified pipeline entry point.
- `requirements.txt`: Python dependencies.

## Setup

1. Install Python 3.
2. Install dependencies:

   `py -m pip install -r requirements.txt`

3. Set the environment variables in PowerShell:

   `$env:OPENWEATHER_API_KEY="YOUR_API_KEY"`

   `$env:WEBHOOK_URL="YOUR_WEBHOOK_RECEIVER_URL/webhook"`

4. Start the receiver from the earlier task in its own terminal.
5. Run the pipeline:

   `py main.py`

6. Stop the repeating pipeline with Ctrl+C.

## Testing

Run the database test:

`py -m unittest discover -s tests`

## Engineering Decisions

- SQLite is used because it is lightweight and requires no separate server.
- API keys and webhook URLs are stored in environment variables.
- The pipeline uses one payload for both database storage and webhook delivery.
- Database queries use parameterized values.
- SQL aggregation groups by both customer ID and name.
- LEFT JOIN filters are placed in the ON clause to preserve customers
  without completed orders.
- Appointment cancellations are retained for history.
- Booking overlap prevention requires application-level validation
  and transaction handling.

## Limitations

- The sample pipeline stores weather observations.
- The webhook receiver must be running and accept JSON POST requests.
- A successful database insert followed by a webhook failure can leave
  a record stored without successful delivery.
- Production systems should add retry handling and duplicate prevention.